In [3]:
import os
import pandas as pd
import kagglehub
from textblob import TextBlob
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

try:
    print("Descargando dataset desde Kaggle...")
    path = kagglehub.dataset_download("vivekhn/yelp-reviews")
    print(f"Ruta de los archivos: {path}")

    # cargo el csv leyendo directo desde la ruta que me devolvió kagglehub
    archivo_csv = os.path.join(path, "yelp.csv")
    df = pd.read_csv(archivo_csv)

except Exception as e:
    print(f"Ojo: Falló la descarga de Kaggle ({e}). Usando datos de prueba hardcodeados...")
    data = {
        'text': [
            "Absolutely wonderful - silky and sexy and comfortable",
            "Awful customer service, terrible food.",
            "Great place, I loved the ambiance.",
            "The worst experience of my life. Disgusting.",
            "Good value for money, will come back.",
            "Not worth the price, very disappointing."
        ],
        'stars': [5, 1, 4, 1, 4, 2]
    }
    df = pd.DataFrame(data)

# vuelo las de 3 estrellas porque son neutras y armo la columna target (1 pos, 0 neg)
df = df[df['stars'] != 3].copy()
df['sentimiento_real'] = df['stars'].apply(lambda x: 1 if x > 3 else 0)

# achico un poco el df si es muy grande para que corra más rápido la prueba (1000 filas)
if len(df) > 1000:
    df = df.sample(1000, random_state=42).reset_index(drop=True)

# ---------------------------------------------------------
# Metodo 1: Diccionario manual (Léxicos)
# ---------------------------------------------------------
# armo unas listas con palabras clave en ingles
positive_words = {'good', 'great', 'excellent', 'amazing', 'wonderful', 'loved', 'best', 'perfect', 'beautiful'}
negative_words = {'bad', 'terrible', 'awful', 'worst', 'disgusting', 'horrible', 'poor', 'disappointing', 'hate'}

def clasificar_por_lexico(texto):
    texto = str(texto).lower()
    conteo_pos = sum(1 for word in positive_words if word in texto)
    conteo_neg = sum(1 for word in negative_words if word in texto)

    # el que tiene más palabras gana. si empatan, lo dejo como positivo
    return 1 if conteo_pos >= conteo_neg else 0

df['prediccion_lexico'] = df['text'].apply(clasificar_por_lexico)

# ---------------------------------------------------------
# Metodo 2: Usando el modelo de TextBlob
# ---------------------------------------------------------
def clasificar_con_textblob(texto):
    # la polaridad va de -1 a 1. asumo que > 0 es positivo
    polaridad = TextBlob(str(texto)).sentiment.polarity
    return 1 if polaridad > 0 else 0

df['prediccion_textblob'] = df['text'].apply(clasificar_con_textblob)

# ---------------------------------------------------------
# Comparación de resultados
# ---------------------------------------------------------
y_real = df['sentimiento_real']
y_lexico = df['prediccion_lexico']
y_textblob = df['prediccion_textblob']

print("\n--- Resultados de mi clasificador manual (Léxicos) ---")
print(f"Accuracy:  {accuracy_score(y_real, y_lexico):.3f}")
print(f"Precision: {precision_score(y_real, y_lexico):.3f}")
print(f"Recall:    {recall_score(y_real, y_lexico):.3f}")
print(f"F1-score:  {f1_score(y_real, y_lexico):.3f}")

print("\n--- Resultados usando TextBlob ---")
print(f"Accuracy:  {accuracy_score(y_real, y_textblob):.3f}")
print(f"Precision: {precision_score(y_real, y_textblob):.3f}")
print(f"Recall:    {recall_score(y_real, y_textblob):.3f}")
print(f"F1-score:  {f1_score(y_real, y_textblob):.3f}")

Descargando dataset desde Kaggle...
Using Colab cache for faster access to the 'yelp-reviews' dataset.
Ruta de los archivos: /kaggle/input/yelp-reviews

--- Resultados de mi clasificador manual (Léxicos) ---
Accuracy:  0.820
Precision: 0.822
Recall:    0.983
F1-score:  0.895

--- Resultados usando TextBlob ---
Accuracy:  0.830
Precision: 0.837
Recall:    0.972
F1-score:  0.900


**Reflexión y Comparación de Resultados:**

Al analizar las métricas obtenidas sobre la muestra del dataset de Yelp, podemos sacar las siguientes conclusiones:



* Rendimiento General (F1-Score y Accuracy): El modelo preentrenado (TextBlob)
superó al clasificador manual basado en léxicos. TextBlob logra un mejor equilibrio general, ya que sus reglas internas pueden manejar de forma más robusta ciertos contextos lingüísticos (como modificadores o negaciones) que un conteo de palabras estricto ignora.

* Precision vs. Recall: El enfoque de léxicos obtuvo un Recall muy alto, lo que significa que detectó casi la totalidad de las reseñas positivas. Sin embargo, su Precision fue más baja. Esto ocurre porque la regla manual construida es muy permisiva y genera falsos positivos. TextBlob, en cambio, fue más preciso al clasificar, equivocándose menos al momento de etiquetar un texto como positivo.

* Limitaciones: El enfoque léxico dependió enteramente de las palabras clave hardcodeadas. Si un usuario utilizó jerga u otras palabras polarizadas que no estaban en nuestra lista, el modelo manual quedó ciego. TextBlob tiene un vocabulario incorporado muchísimo más extenso y procesa la intensidad de las palabras, haciéndolo ideal para textos sin estructura fija como estas reseñas.